In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('../Data/당뇨.csv')
df.head()

,임신횟수,혈당,혈압,피부두께,인슐린,BMI,가족력지표,나이,당뇨
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
df.당뇨.value_counts()

당뇨
0    500
1    268
Name: count, dtype: int64

> 당뇨 환자는 268, 당뇨 아닌 환자는 500으로 사용 가능한 데이터로 판단


In [4]:
df.describe()

,임신횟수,혈당,혈압,피부두께,인슐린,BMI,가족력지표,나이,당뇨
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


> min값이 0인 것이 상당히 많음.                 
> 임신횟수와 당뇨는 0이 있어도 상관없으나 나머지는 이상치 또는 결측치로 추측                    

In [5]:
df.isnull().sum()

임신횟수     0
혈당       0
혈압       0
피부두께     0
인슐린      0
BMI      0
가족력지표    0
나이       0
당뇨       0
dtype: int64

> null은 없는 것으로 파악 되었으니 0이 결측치로 봐도 무방하다고 판단

In [6]:
# 이상치라 판단되면 삭제하는 함수

def drop_outliers(df, column):
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = (q3 - q1) * 1.5
    
    lower_bound = q1 - iqr
    upper_bound = q3 + iqr
    
    mask = (df[column] > 0) & ((df[column] < lower_bound) | (df[column] > upper_bound))
    outlier_indices = df[mask].index
    return df.drop(outlier_indices)

In [7]:

for col in ['혈당', 'BMI', '인슐린', '혈압','피부두께']:
    df = drop_outliers(df, col)

df.당뇨.value_counts()

# 

당뇨
0    479
1    238
Name: count, dtype: int64

> 위의 65:35와 얼추 비슷한 비율이므로 사용할 수 있겠다고 판단 내림

In [8]:
# 0인 결측치를 당뇨 환자와 아닌 사람의 그룹으로 구분 후, 각 컬럼의 중위값으로 대치
cols_to_fix = ['혈당', '혈압', '피부두께', '인슐린', 'BMI']
df[cols_to_fix] = df[cols_to_fix].replace(0, np.nan)

for col in cols_to_fix:
    df[col] = df[col].fillna(df.groupby('당뇨')[col].transform('median'))


In [9]:
df.to_csv('../Data/당뇨_전처리.csv',index=None)